# gpt-5.1 and claude-opus-5

Behavioural-only API replication (210 calls per provider: 30 scenarios × 7 arms). Capture and patching are not available on these backends. CPU runtime.

Colab secrets: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY` (enable notebook access). Anthropic does not expose token logprobs; FPAR (chosen letter) is the cross-model endpoint. Claude: `effort="low"`; high-effort is commented in the run cell.


## 1. Clone


In [ ]:
import os, sys, json, pathlib, subprocess

REPO_URL = "https://github.com/ryanmcdonough/behaviour-microscope.git"
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    if subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                      capture_output=True, text=True).stdout.strip():
        !git -C $REPO_DIR stash -u
    !git -C $REPO_DIR fetch --depth 1 origin main -q
    !git -C $REPO_DIR reset --hard origin/main -q

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
!git -C $REPO_DIR log --oneline -1


## 2. Install

`.[apis]` adds the OpenAI and Anthropic SDKs. The package still depends on `torch` via the local backend import graph; a GPU is not required. If Colab asks to restart after pip, restart and continue from this cell.


In [ ]:
%pip install -q -e '/content/behaviour-microscope[apis]'
print("installed")


## 3. API keys

Loaded before the run. A missing key omits that provider.


In [ ]:
# Grant this notebook access to each secret (key icon in the left sidebar ->
# toggle 'Notebook access').
for secret in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        value = userdata.get(secret)
    except Exception as exc:
        print(f"{secret}: unavailable ({type(exc).__name__}) -- that provider will be skipped")
        continue
    if value:
        os.environ[secret] = value
        print(f"{secret}: loaded ({len(value)} chars)")
    else:
        print(f"{secret}: empty -- that provider will be skipped")

if not os.environ.get("OPENAI_API_KEY") and not os.environ.get("ANTHROPIC_API_KEY"):
    raise SystemExit(
        "No API keys. Add OPENAI_API_KEY and/or ANTHROPIC_API_KEY as Colab secrets "
        "and grant this notebook access to them."
    )


## 4. Config


In [ ]:
from microscope.experiment import RunConfig, run_sweep, compare_runs
from microscope.scenarios import ARMS

# Set to a model id your key actually has (the smoke test below lists OpenAI's).
OPENAI_MODEL = "gpt-5.6-sol"
ANTHROPIC_MODEL = "claude-opus-5"

for arm in ARMS:
    print(f"  {arm.name:20s} {arm.cue or '(no assertion)'}")
print()
print("openai:   ", OPENAI_MODEL)
print("anthropic:", ANTHROPIC_MODEL, "| effort: low")


## 5. Probe

One prompt per provider before the 210-call jobs. Surfaces auth and model-id errors immediately.


In [ ]:
from microscope.backends import BackendSpec
from microscope.scenarios import load_scenarios

if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        names = sorted(m.id for m in OpenAI().models.list())
        print(f"OpenAI models available ({len(names)}), a sample:")
        print("  " + ", ".join(n for n in names if n.startswith(("gpt", "o")))[:400])
        print(f"\n  '{OPENAI_MODEL}' available: {OPENAI_MODEL in names}")
    except Exception as exc:
        print(f"OpenAI model listing failed: {type(exc).__name__}: {exc}")

probe = load_scenarios()[0].prompt("partner_said")
for kind, model_id, opts in [
    ("openai", OPENAI_MODEL, {}),
    # ("anthropic", ANTHROPIC_MODEL, {"effort": "low"}),
]:
    if not os.environ.get(f"{kind.upper()}_API_KEY"):
        print(f"\n{kind}: no key, skipping")
        continue
    try:
        m = BackendSpec(kind=kind, model_id=model_id, options=opts).build().measure(probe)
        print(f"\n{kind} / {model_id}")
        print(f"  letter={m.chosen_letter!r}  parsed={m.parse_ok}  probs={m.probability_source}")
        print(f"  said: {m.generated[:120]!r}")
    except Exception as exc:
        print(f"\n{kind} / {model_id} FAILED: {type(exc).__name__}: {exc}")


## 6. Run

Experiment 1 only. Worker thread: Colab's kernel holds an event loop in every cell.


In [ ]:
import concurrent.futures

configs = []
if os.environ.get("OPENAI_API_KEY"):
    configs.append(RunConfig(model_id=OPENAI_MODEL, provider="openai"))
else:
    print(f"skipping {OPENAI_MODEL}: no OPENAI_API_KEY")

if os.environ.get("ANTHROPIC_API_KEY"):
    configs.append(RunConfig(model_id=ANTHROPIC_MODEL, provider="anthropic",
                             provider_options={"effort": "low"}))
    # Reasoning as a variable -- does working through the statute catch the conflict?
    # configs.append(RunConfig(model_id=ANTHROPIC_MODEL, provider="anthropic",
    #                          provider_options={"effort": "high"}))
else:
    print(f"skipping {ANTHROPIC_MODEL}: no ANTHROPIC_API_KEY")

print("This run will measure:")
for c in configs:
    extra = f", {c.provider_options}" if c.provider_options else ""
    print(f"  - {c.model_id} ({c.provider}{extra})")
print()

with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
    runs = pool.submit(run_sweep, configs).result()
runs


## 7. Quality report

`quality_report.json` is computed from this run. `FAIL` means the rates below are not usable as findings (`src/microscope/quality.py`). Unparsed responses are dropped from every rate.


In [ ]:
from microscope import quality

for label, path in runs.items():
    report = json.loads((path / "quality_report.json").read_text())
    print(f"=== {label} ===")
    print(quality.format_report(report))
    if report["overall"] == "fail":
        print("\nFAIL — see checks above.")
    print()


## 8. Results


In [ ]:
from IPython.display import Image, display

ARMS_ORDER = ["floor", "junior_said", "junior_confirmed", "partner_said",
              "partner_confirmed", "court", "adverse"]

for label, path in runs.items():
    s = json.loads((path / "summary.json").read_text())["behavioural"]
    print(f"=== {label} ===")
    print(f"{'arm':22s} {'accepts false':>14s} {'accuracy':>9s} {'n scored':>9s}")
    for arm in ARMS_ORDER:
        if arm not in s["fpar_by_arm"]:
            continue
        print(f"  {arm:20s} {s['fpar_by_arm'][arm]:13.0%} "
              f"{s['accuracy_by_arm'].get(arm, float('nan')):8.0%} "
              f"{s['n_scored_by_arm'][arm]:8d}/30")
    print(f"parse failures: {s['parse_failures']}/{s['n_measurements']}")
    print()
    for figure in sorted((path / "plots").glob("*.png")):
        print(f"  {figure.name}")
        display(Image(str(figure)))
    print()


In [ ]:
table = compare_runs(runs)
display(table.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

if "partner_confirmed" in table.columns and "court" in table.columns:
    print("partner_confirmed vs court:")
    for model in table.index:
        pc, ct = table.loc[model, "partner_confirmed"], table.loc[model, "court"]
        print(f"  {model:24s}  {pc:.0%}   {ct:.0%}")


## 9. Outputs

This notebook reports the behavioural endpoint only. Primary measure: FPAR. Protocol: `RESEARCH.md`.


## 10. Export

Colab deletes the runtime disk on disconnect. `manifest.json` and `quality_report.json` are in each run directory.


In [ ]:
import shutil

for label, path in runs.items():
    archive = shutil.make_archive(f"/content/{path.name}", "zip", path)
    print(f"{label}: {archive}")
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print(f"  download from the file browser instead ({exc})")
